# **MICHAEL SCOTT TRADING COMPANY**

**INTRODUCTION:**

We've built a NN model that'll analyze btc market during 2018-2023,

then used our own backtesting engine which is used to enforce our risk management strategies



------------------------------------------------------------------------------------------------------------------------------------------------




*   **IMPORTING LIBRARIES FOR OUR NN MODEL**
*   **LOADING OUR DATASET FEATURE ENGINEERING AND DATA PREPROCESSING**


In [ ]:
# Basic Python, Scikit and Pytorch Libraries to help build, train and test our model
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
import random
# NOTE: These are not backtesting libraries

# LOADING OUR DATASET
df1 = pd.read_csv('2018-2023_BTC_dataset.csv')

# We've have progressively changed our training data
# Eg: we've first trained our model on 2018 and 2019 btc dataset then tested on 2020 dataset
# Then, for testing 2021 we've retrained our model with 2018, 2019 and 2020 dataset
# Since, If we are in year 2021 we would have the data for the year 2020 to re-train our model

# 1. SPLITTING DATE INTO SEPARATE COLUMNS
df1['Date'] = pd.to_datetime(df1['Date'], format='%d-%m-%Y')
df1['Year'] = df1['Date'].dt.year
df1['Target'] = (df1['Close'].shift(-5) > df1['Close']).astype(int)

# 2. THE MACRO TREND FILTER
df1['SMA_50'] = df1['Close'].rolling(window=50).mean()

# 3. Stationary Price & Volume Features
df1['Returns'] = df1['Close'].pct_change() * 100
df1['Volume_Change'] = df1['Volume'].pct_change() * 100
df1['range_pct'] = ((df1['High'] - df1['Low']) / df1['Close']) * 100
df1['close-high'] = (df1['Close'] - df1['High'])/(df1['Close']) * 100
df1['close-low'] = (df1['Close'] - df1['Low'])/(df1['Close']) * 100
df1['day_change_pct'] = ((df1['Close'] - df1['Open']) / df1['Open']) * 100

# 4. MACD Histogram
df1['ema12'] = df1['Close'].ewm(span=12, adjust=False).mean()
df1['ema26'] = df1['Close'].ewm(span=26, adjust=False).mean()
df1['macd'] = df1['ema12'] - df1['ema26']
df1['SignalLine'] = df1['macd'].ewm(span=9, adjust=False).mean()
df1['MACD_Hist'] = df1['macd'] - df1['SignalLine']

# 5. RSI
delta = df1['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df1['RSI'] = 100 - (100 / (1 + rs))

# 6. Bollinger Bands Width
df1['SMA_20'] = df1['Close'].rolling(window=20).mean()
df1['STD_20'] = df1['Close'].rolling(window=20).std()
df1['BB_Width'] = (df1['STD_20'] * 4) / df1['SMA_20']

# 7. Normalized ATR & Regime
df1['prev_close'] = df1['Close'].shift(1)
df1['TR'] = df1.apply(lambda row: max(
    row['High'] - row['Low'],
    abs(row['High'] - row['prev_close']),
    abs(row['Low'] - row['prev_close'])
) if not pd.isna(row['prev_close']) else row['High'] - row['Low'], axis=1)
df1['ATR_14'] = df1['TR'].rolling(window=14).mean()
df1['ATR_pct'] = (df1['ATR_14'] / df1['Close']) * 100

df1['Vol_20'] = df1['Returns'].rolling(window=20).std()
df1['Vol_100'] = df1['Returns'].rolling(window=100).std()
df1['Regime'] = (df1['Vol_20'] > df1['Vol_100']).astype(int)

# 8. ADX (Average Directional Index)
df1['UpMove'] = df1['High'] - df1['High'].shift(1)
df1['DownMove'] = df1['Low'].shift(1) - df1['Low']
df1['+DM'] = np.where((df1['UpMove'] > df1['DownMove']) & (df1['UpMove'] > 0), df1['UpMove'], 0)
df1['-DM'] = np.where((df1['DownMove'] > df1['UpMove']) & (df1['DownMove'] > 0), df1['DownMove'], 0)
df1['TR_14'] = df1['TR'].ewm(span=14, adjust=False).mean()
df1['+DI_14'] = 100 * (df1['+DM'].ewm(span=14, adjust=False).mean() / df1['TR_14'])
df1['-DI_14'] = 100 * (df1['-DM'].ewm(span=14, adjust=False).mean() / df1['TR_14'])
df1['DX'] = 100 * (abs(df1['+DI_14'] - df1['-DI_14']) / (df1['+DI_14'] + df1['-DI_14']))
df1['ADX_14'] = df1['DX'].ewm(span=14, adjust=False).mean()

# 9. Stationary OBV Oscillator
df1['OBV_raw'] = (np.sign(df1['Close'].diff()) * df1['Volume']).fillna(0).cumsum()
df1['OBV_SMA_20'] = df1['OBV_raw'].rolling(window=20).mean()
df1['OBV_STD_20'] = df1['OBV_raw'].rolling(window=20).std()
df1['OBV_Oscillator'] = (df1['OBV_raw'] - df1['OBV_SMA_20']) / df1['OBV_STD_20']

# 10. #   NaNs CREATED BY OUR ROLLING INDICATORS AND SHIFTS
df1 = df1.replace([np.inf, -np.inf], np.nan).dropna()

# 11. FEATURE LIST
features = ['Returns', 'Volume_Change', 'range_pct', 'close-high',
            'close-low', 'day_change_pct', 'MACD_Hist', 'RSI',
            'BB_Width', 'ATR_pct', 'ADX_14', 'OBV_Oscillator']
target_col_name = 'Target'


*   **BUILDING OUR MODEL**
*   **MODEL TRAINING AND TESTING**


In [ ]:

# 1. DEFINING THE PYTORCH MODEL
class TradingModel(nn.Module):
    def __init__(self, h1=64, h2=32, h3=16):
        super().__init__()
        self.fc1 = nn.LazyLinear(h1)
        self.fc2 = nn.LazyLinear(h2)
        self.drop2 = nn.Dropout(0.2)
        self.fc3 = nn.LazyLinear(h3)
        self.drop3 = nn.Dropout(0.2)
        self.out = nn.LazyLinear(2)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.drop2(x)
        x = F.relu(self.fc3(x))
        x = self.drop3(x)
        return self.out(x)

# 2. INITIALISING TRACKING ARRAYS
all_test_prices = []
all_test_probs = []
all_targets = []
all_test_atrs = []
all_test_regimes = []
all_test_sma50s = []
all_test_adxs = []
test_years = [2020, 2021, 2022, 2023]
num_experts = 15
seed = 1

for year in test_years:
    print(f"\n{'='*40}")
    print(f"  PROCESSING YEAR: {year} (Training {num_experts} Models)")
    print(f"{'='*40}")

    train_df = df1[df1['Date'].dt.year < year].dropna()
    test_df = df1[df1['Date'].dt.year == year].dropna()

    if train_df.empty:
        continue

    scaler = StandardScaler()
    X_train_raw = scaler.fit_transform(train_df[features])
    X_test_raw = scaler.transform(test_df[features])

    X_train = torch.FloatTensor(X_train_raw)
    y_train = torch.LongTensor(train_df['Target'].values)
    X_test = torch.FloatTensor(X_test_raw)

    year_expert_probs = []

    for i in range(num_experts):
        if (i + 1) % 5 == 0 or i == 0:
            print(f" -> Training Expert {i+1}/{num_experts}...")

        expert_seed = seed + i
        random.seed(expert_seed)
        np.random.seed(expert_seed)
        torch.manual_seed(expert_seed)

        model = TradingModel(h1=64, h2=32, h3=16)
        criterion = nn.CrossEntropyLoss()

        _ = model(X_train[0:1])
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

        model.train()
        for epoch in range(50):
            optimizer.zero_grad()
            loss = criterion(model(X_train), y_train)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            probs = F.softmax(model(X_test), dim=1)[:, 1].numpy()
            year_expert_probs.append(probs)

    print(f" [*] Averaging {num_experts} experts for {year}...")
    ensemble_probs = np.mean(year_expert_probs, axis=0)

# 3. CAPTURING ALL DATA FOR THE BACKTESTER
    all_test_probs.extend(ensemble_probs)
    all_test_prices.extend(test_df['Close'].values)
    all_targets.extend(test_df['Target'].values)
    all_test_atrs.extend(test_df['ATR_14'].values)
    all_test_regimes.extend(test_df['Regime'].values)
    all_test_sma50s.extend(test_df['SMA_50'].values)
    all_test_adxs.extend(test_df['ADX_14'].values)


  PROCESSING YEAR: 2020 (Training 15 Models)
 -> Training Expert 1/15...
 -> Training Expert 5/15...
 -> Training Expert 10/15...
 -> Training Expert 15/15...
 [*] Averaging 15 experts for 2020...

  PROCESSING YEAR: 2021 (Training 15 Models)
 -> Training Expert 1/15...
 -> Training Expert 5/15...
 -> Training Expert 10/15...
 -> Training Expert 15/15...
 [*] Averaging 15 experts for 2021...

  PROCESSING YEAR: 2022 (Training 15 Models)
 -> Training Expert 1/15...
 -> Training Expert 5/15...
 -> Training Expert 10/15...
 -> Training Expert 15/15...
 [*] Averaging 15 experts for 2022...

  PROCESSING YEAR: 2023 (Training 15 Models)
 -> Training Expert 1/15...
 -> Training Expert 5/15...
 -> Training Expert 10/15...
 -> Training Expert 15/15...
 [*] Averaging 15 experts for 2023...



*   **BUILDING OUR BACKTESTING ENGINE AND EXECUTION**



In [ ]:
class AdaptiveBacktester:
    def __init__(self, initial_balance=10000, fee=0.0015):
        self.initial_balance = initial_balance
        self.balance = initial_balance
        self.fee = fee
        self.inventory = 0
        self.buy_price = 0
        self.highest_price = 0
        self.cash = initial_balance
        self.invested = 0
        self.equity_curve = [initial_balance]
        self.trades = []
        self.gross_profit = 0
        self.gross_loss = 0

    def run(self, prices, probabilities, atrs, regimes, sma50s, adxs):
        for i in range(1, len(prices) - 1):
            current_price = prices[i]
            tomorrow_price = prices[i + 1]
            current_atr = atrs[i]
            current_regime = regimes[i]
            current_sma50 = sma50s[i]
            current_adx = adxs[i]
            if self.inventory == 1:
                if current_price > self.highest_price:
                    self.highest_price = current_price

                # Wider stops for multi-day holding
                atr_multiplier = 2.5 if current_regime == 1 else 4.0
                stop_loss_floor = self.highest_price - (current_atr * atr_multiplier)

                # 1. STOP-LOSS HIT
                if current_price <= stop_loss_floor:
                    sell_value = self.invested * (stop_loss_floor / self.buy_price)
                    sell_value *= (1 - self.fee)
                    self._record_trade(sell_value)
                    self._reset_position(sell_value)

                # 2. TREND REVERSAL
                elif probabilities[i] < 0.45 or current_price < current_sma50:
                    sell_value = self.invested * (tomorrow_price / self.buy_price)
                    sell_value *= (1 - self.fee)
                    self._record_trade(sell_value)
                    self._reset_position(sell_value)

            if self.inventory == 0:
                entry_threshold = 0.55 if current_regime == 1 else 0.52

                if probabilities[i] > entry_threshold and current_price > current_sma50:
                    alloc_pct = 0.5 if current_regime == 1 else 0.8
                    amount_to_invest = self.cash * alloc_pct
                    self.invested = amount_to_invest * (1 - self.fee)
                    self.cash -= amount_to_invest
                    self.inventory = 1
                    self.buy_price = tomorrow_price
                    self.highest_price = tomorrow_price

            if self.inventory == 1:
                current_val = self.cash + (self.invested * (current_price / self.buy_price))
            else:
                current_val = self.cash
            self.equity_curve.append(current_val)

        return self.equity_curve[-1]

    def _record_trade(self, sell_value):
        initial_investment = self.invested / (1 - self.fee)
        trade_pnl = sell_value - initial_investment
        self.trades.append(trade_pnl)
        if trade_pnl > 0:
            self.gross_profit += trade_pnl
        else:
            self.gross_loss += abs(trade_pnl)

    def _reset_position(self, sell_value):
        self.cash += sell_value
        self.invested = 0
        self.inventory = 0
        self.highest_price = 0

    def calculate_metrics(self, btc_prices):
        eq = np.array(self.equity_curve)
        returns = (eq[1:] - eq[:-1]) / eq[:-1]
        net_profit = eq[-1] - eq[0]
        peak = np.maximum.accumulate(eq)
        drawdown = (eq - peak) / peak
        max_dd = np.min(drawdown)

        mean_return = np.mean(returns)
        std_dev = np.std(returns)
        sharpe = (mean_return / std_dev * np.sqrt(252)) if std_dev != 0 else 0

        downside_returns = returns[returns < 0]
        sortino_std = np.std(downside_returns) if len(downside_returns) > 0 else 0
        sortino = (mean_return / sortino_std * np.sqrt(252)) if sortino_std != 0 else 0

        total_trades = len(self.trades)
        winning_trades = len([t for t in self.trades if t > 0])
        win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0
        buy_and_hold_return = ((btc_prices[-1] - btc_prices[0]) / btc_prices[0]) * 100

        return {
            "1. Profitability & Returns": "",
            "Gross Profit": f"${self.gross_profit:,.2f}",
            "Gross Loss": f"${self.gross_loss:,.2f}",
            "Net Profit": f"${net_profit:,.2f}",
            "Buy & Hold Return": f"{buy_and_hold_return:.2f}%",
            "Total Strategy Return": f"{(net_profit / self.initial_balance) * 100:.2f}%",
            "\n2. Risk & Volatility": "",
            "Max Drawdown": f"{max_dd * 100:.2f}%",
            "Sharpe Ratio": f"{sharpe:.2f}",
            "Sortino Ratio": f"{sortino:.2f}",
            "\n3. Trade Statistics": "",
            "Total Closed Trades": total_trades,
            "Win Rate": f"{win_rate:.2f}%"
        }

# 3. EXECUTION
engine = AdaptiveBacktester(initial_balance=10000)
engine.run(np.array(all_test_prices), np.array(all_test_probs), np.array(all_test_atrs), np.array(all_test_regimes), np.array(all_test_sma50s), np.array(all_test_adxs))
results = engine.calculate_metrics(all_test_prices)

for k, v in results.items():
    print(f"{k}: {v}" if v != "" else f"{k}")

1. Profitability & Returns
Gross Profit: $51,104.71
Gross Loss: $31,017.96
Net Profit: $20,086.74
Buy & Hold Return: 485.86%
Total Strategy Return: 200.87%

2. Risk & Volatility
Max Drawdown: -24.84%
Sharpe Ratio: 0.94
Sortino Ratio: 1.01

3. Trade Statistics
Total Closed Trades: 129
Win Rate: 34.11%




**RESULTS**



...........................................................


**i) Profitability & Returns**


Gross Profit: $51,104.71


Gross Loss: $31,017.96


Net Profit: $20,086.74


Buy & Hold Return: 485.86%


Total Strategy Return: 200.87%



...........................................................




**ii) Risk & Volatility**


Max Drawdown: -24.84%


Sharpe Ratio: 0.94


Sortino Ratio: 1.01



...........................................................



**iii) Trade Statistics**



Total Closed Trades: 129


Win Rate: 34.11%


...........................................................